In [35]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy import stats
from scipy.stats import zscore
from scipy.stats.mstats import winsorize

In [36]:
# Load Dataset

df_original = pd.read_csv("patient_health_data.csv")

df = df_original.copy()

### PART A: HANDLING MISSING VALUES

In [37]:
# 1. Missing Value Summary Report

missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().mean() * 100)
})

print("\nMissing Value Summary:\n", missing_summary)


Missing Value Summary:
                 Missing_Count  Missing_Percentage
patient_id                  0                 0.0
age                        20                10.0
gender                     20                10.0
region                     20                10.0
bmi                        20                10.0
blood_pressure              0                 0.0
cholesterol                18                 9.0
glucose                    20                10.0
disease_risk                0                 0.0


In [38]:
# 2. Imputation Techniques

# Simple Imputer (Numerical)
mean_imputer = SimpleImputer(strategy="mean")
df["bmi"] = mean_imputer.fit_transform(df[["bmi"]])

# Simple Imputer (Categorical)
cat_imputer = SimpleImputer(strategy="most_frequent")
df["region"] = cat_imputer.fit_transform(df[["region"]]).ravel()

# Most Frequent Imputation (Gender)
gender_imputer = SimpleImputer(strategy="most_frequent")
df["gender"] = gender_imputer.fit_transform(df[["gender"]]).ravel()

# Missing Indicator + Random Sample Imputation
for col in ["cholesterol", "glucose"]:
    df[col + "_missing"] = df[col].isnull().astype(int)

    random_sample = df[col].dropna().sample(df[col].isnull().sum(), replace=True)
    random_sample.index = df[df[col].isnull()].index

    df.loc[df[col].isnull(), col] = random_sample

# KNN Imputer
knn_cols = ["age", "blood_pressure", "cholesterol", "glucose"]
knn_imputer = KNNImputer(n_neighbors=5)
df[knn_cols] = knn_imputer.fit_transform(df[knn_cols])

# MICE (Iterative Imputer)
mice_imputer = IterativeImputer(max_iter=10, random_state=0)
df[knn_cols] = mice_imputer.fit_transform(df[knn_cols])

### PART B: Handling Outliers

In [39]:
import warnings

warnings.filterwarnings("ignore")

df_before_outliers = df.copy()

# Z-score
z_cols = ["cholesterol", "glucose"]
z_scores = np.abs(zscore(df[z_cols]))
df = df[(z_scores < 3).all(axis=1)]

# IQR
Q1 = df["bmi"].quantile(0.25)
Q3 = df["bmi"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df = df[(df["bmi"] >= lower_bound) & (df["bmi"] <= upper_bound)]

# Percentile Capping
for col in ["bmi", "blood_pressure", "cholesterol", "glucose"]:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = np.clip(df[col], lower, upper)

# Winsorization
for col in ["bmi", "blood_pressure", "cholesterol", "glucose"]:
    df[col] = np.array(winsorize(df[col], limits=[0.01, 0.01]))

In [40]:
print("Before Outliers Shape:", df_before_outliers.shape)
print("After Outliers Shape:", df.shape)

print("\n--- BEFORE OUTLIER TREATMENT ---")
print(df_before_outliers.describe())

print("\n--- AFTER OUTLIER TREATMENT ---")
print(df.describe())

Before Outliers Shape: (200, 11)
After Outliers Shape: (178, 11)

--- BEFORE OUTLIER TREATMENT ---
       patient_id         age         bmi  blood_pressure  cholesterol  \
count  200.000000  200.000000  200.000000      200.000000   200.000000   
mean   100.500000   48.910000   25.588667      123.748516   202.433250   
std     57.879185   17.918989    6.405076       25.234601    42.931857   
min      1.000000   18.000000   10.000000       82.930000   100.000000   
25%     50.750000   34.000000   22.000000      109.920000   182.460000   
50%    100.500000   49.100000   25.588667      120.080000   199.850000   
75%    150.250000   62.000000   28.435000      130.260000   221.455000   
max    200.000000   79.000000   50.000000      247.649217   350.000000   

          glucose  disease_risk  cholesterol_missing  glucose_missing  
count  200.000000    200.000000             200.0000       200.000000  
mean   114.446103      0.525000               0.0900         0.100000  
std     49.578836 

### Part C: Final Clean Dataset

In [41]:
# Final Clean Dataset

print("Final Missing Values:\n", df.isnull().sum())
print("Final Dataset Shape:", df.shape)
print("\nPreview:\n", df.head())
df.to_csv("final_cleaned_patient_data.csv", index=False)

Final Missing Values:
 patient_id             0
age                    0
gender                 0
region                 0
bmi                    0
blood_pressure         0
cholesterol            0
glucose                0
disease_risk           0
cholesterol_missing    0
glucose_missing        0
dtype: int64
Final Dataset Shape: (178, 11)

Preview:
    patient_id   age  gender region    bmi  blood_pressure  cholesterol  \
0           1  56.0  Female   West  14.87      144.430000       100.00   
2           3  46.0    Male  South  21.69       94.450000       161.30   
3           4  32.0    Male   West  29.26      119.170000       161.15   
4           5  60.0    Male  South  21.04      125.760000       189.93   
5           6  51.2    Male  South  24.43      216.006408       250.07   

   glucose  disease_risk  cholesterol_missing  glucose_missing  
0   131.01             1                    0                0  
2   119.69             1                    0                0  
3    95

#### **Brief Report**

**1. Imputation Strategy:**

- Numerical feature BMI was imputed using mean, which maintains overall distribution.
- Categorical features Region and Gender were imputed using the most frequent value to preserve category consistency.
- Advanced methods such as KNN Imputation and MICE were applied to handle multivariate missing values, ensuring realistic and relationship-based imputation.

*Most Effective Method:*
MICE was found to be the most effective as it considers relationships between multiple variables and produces more accurate estimations.

**2. Outlier Handling Strategy:**

- Z-score method removed extreme values in cholesterol and glucose.
- IQR method removed unusual BMI values.
- Percentile capping limited extreme values.
- Winsorization capped outliers without removing data.

*Best Method:*
Winsorization preserved data size while controlling extreme values, making it the most suitable method for maintaining data quality.

**3. Improvement in Dataset:**

- All missing values were successfully handled.
- Extreme outliers were removed or capped.
- Data distribution became more stable.
- Noise and inconsistencies were reduced.

Result:
The dataset is now clean, consistent, and suitable for machine learning models.

In [42]:
# Profiling Summary

from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Final Clean Dataset Report", explorative=True)
profile.to_file("final_profile_report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 202.56it/s]
